# GEMS G12 — Deep Geological Signature Discovery

**End-to-end:** original GeoDAWN rasters → mask-aware physical features → geological EDA → 6.4 km embargoed geography splits → feature-family ablations → official-formula **offline known-fault proxy** DTI → evidence atlas → strict submission GeoTIFF.

**Read before running:** Offline known-fault DTI is *not* the competition's private new-fault DTI. No legacy G11 recovery; no manual extraction of 651-layer feature caches. The notebook uses an existing repository module, with cached feature rasters and atomic writes. It does **not** upload a scored submission automatically.

In [ ]:
import os, sys, subprocess, importlib.util, json, time
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/GEMS_G12_MAIN')
URL='https://github.com/mybenkhadda/GEMS---drivendata.git'
if not (ROOT/'.git').exists():
    subprocess.run(['git','clone','--depth','1','--branch','main',URL,str(ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'fetch','--depth','1','origin','main'],check=True)
    subprocess.run(['git','-C',str(ROOT),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','xgboost>=2.0','rasterio>=1.3','scipy>=1.10','scikit-learn>=1.3','matplotlib>=3.7','pandas>=2'],check=True)
sys.path.insert(0,str(ROOT))
from gems_bench.g12.deep import build,masks,ablations,error_atlas,submit,ensure_grid
from gems_bench.metric import dti_components,OFFICIAL_EXCLUDE_SUPPORTS_TP
import rasterio, numpy as np, pandas as pd
print('Code commit:',subprocess.check_output(['git','-C',str(ROOT),'rev-parse','--short','HEAD'],text=True).strip())

## 00 — Configuration and exact input discovery

This cell searches only the specified GEMS working directory for the **exact** official filenames. If a filename appears multiple times, set its path explicitly rather than picking a random file.

In [ ]:
DRIVE=Path('/content/drive/MyDrive/GEMS_2')
CFG={
 'raw':None,
 'known':None,
 'template':None,
 'out':str(DRIVE/'outputs/G12_DEEP_DISCOVERY'),
 'seed':2026,
 'groups':('raw','mag','grav','cross','all'),
 'cap':80000,
 'split_block':256,  # 25.6km blocks
 'embargo':64,      # 6.4km gap from training to validation/test
 'n_estimators':280,
}
assert DRIVE.is_dir(),f'Missing {DRIVE}'
def locate(filename,manual):
    if manual:
        p=Path(manual)
        if not p.is_file():raise FileNotFoundError(p)
        return p
    targets=[]
    for here,dirs,files in os.walk(DRIVE):
        # Cached output or archived versions should NOT masquerade as originals.
        dirs[:]=[d for d in dirs if d not in ('outputs','.git','archive','backups')]
        if filename in files: targets.append(Path(here)/filename)
    if len(targets)!=1:
        raise RuntimeError(f'Expected one {filename}, found {len(targets)}: {targets[:10]}. Set CFG path explicitly.')
    return targets[0]
RAW=locate('gems-geodawn-numerical-features.tif',CFG['raw'])
KNOWN=locate('existing_faults.tif',CFG['known'])
TEMPLATE=locate('example_submission.tif',CFG['template'])
OUT=Path(CFG['out']);OUT.mkdir(parents=True,exist_ok=True)
with rasterio.open(RAW) as r,rasterio.open(KNOWN) as k,rasterio.open(TEMPLATE) as t:
    ensure_grid(r,k,t)
    print('Grid:',r.width,r.height,'bands:',r.count,'CRS:',r.crs,'resolution:',r.res)
    print('Available raw bands:',r.descriptions)
print('Raw:',RAW, '| Labels:',KNOWN, '| Template:',TEMPLATE, '| Outputs:',OUT)

## 01 — Label-free, multiscale geological signatures

Features: local anomalies, structural gradients, tensor coherence, Hessian/Laplacian signatures, tilt-angle proxy, directional cross-contact asymmetry, axial strike persistence, magnetic–gravity/conductivity/terrain alignment, local source coverage. Mask-normalized convolution and a 96-cell halo reduce nodata and tile-boundary artifacts. No supervision is used here.

In [ ]:
t0=time.monotonic()
MAN=build(RAW,OUT/'physics_g12.tif',block=256,halo=96)
print('Features:',len(MAN['names']),'Families:',pd.Series(MAN['groups']).value_counts().to_dict())
print('Elapsed seconds:',round(time.monotonic()-t0,1))
print('Cache manifest:',OUT/'physics_g12.manifest.json')

## 02 — Spatial validation diagnostics

Training data is separated by a 6.4 km pixel embargo from validation and test blocks. The **test blocks are not used for model selection**. Known faults in validation/test are proxies for unseen faults, *not verified new-fault ground truth*.

In [ ]:
S=masks(RAW,KNOWN,TEMPLATE,seed=CFG['seed'],block=CFG['split_block'],embargo=CFG['embargo'])
for k in ('train','val','test'):
 print(k,'pixels',int(S[k].sum()),'known fault pixels',int((S[k]&S['faults']).sum()))
assert not np.any(S['train'] & (S['val'] | S['test']))
assert not np.any(S['val'] & S['test'])

## 03 — Feature-family ablation tournament

Run raw-only, then raw + magnetic, gravity, cross-modal, and all physical signatures under identical training budgets. The reported buffered-target PR-AUC is sampled and diagnostic; the continuous offline DTI is computed on the full held-out validation geography. No private fault labels are available.

In [ ]:
t0=time.monotonic()
ROWS=ablations(RAW,OUT/'physics_g12.tif',KNOWN,TEMPLATE,OUT,
    seed=CFG['seed'],groups=CFG['groups'],cap=CFG['cap'],
    split_block=CFG['split_block'],embargo=CFG['embargo'],n_estimators=CFG['n_estimators'])
TABLE=pd.DataFrame(ROWS).sort_values('val_dti_known_fault_proxy',ascending=False)
display(TABLE[['model','extra_features','val_pr_auc_buffered_sample','val_dti_known_fault_proxy','val_fp','seconds']])
print('Total seconds:',round(time.monotonic()-t0,1),'Results:',OUT/'ablation_results.json')

## 04 — Geological failure atlas and physical feature contrasts

Interpret unlabelled high-confidence lineaments as **unverified candidate discoveries**, not true positives or false positives. Feature contrast compares near-known traces against distant mapped background and may be confounded by geological context.

In [ ]:
CHAMP=TABLE.iloc[0].to_dict()
ATLAS=error_atlas(RAW,OUT/'physics_g12.tif',KNOWN,TEMPLATE,CHAMP['prediction_file'],OUT/'atlas',
                  seed=CFG['seed'],split_block=CFG['split_block'],embargo=CFG['embargo'])
from IPython.display import display,Image
display(Image(filename=ATLAS['map_file']))
contrast=pd.read_json(ATLAS['contrast_file'])
display(contrast.head(25))

## 05 — One-time geographic test, then submission export

**Important:** The test DTI below is an **offline withheld-known-fault proxy**; it does not estimate official leaderboard DTI reliably. The exported submission is a single-band float32 GeoTIFF matching the official example exactly, with existing mapped-fault pixels zeroed and outside-area values NaN. Upload manually at DrivenData.

In [ ]:
with rasterio.open(CHAMP['prediction_file']) as p:
    final=dti_components(p.read(1),S['faults'],eval_mask=S['test'],
                          exclude_supports_tp=OFFICIAL_EXCLUDE_SUPPORTS_TP).as_dict()
print('Held-out geographic test — known-fault DTI PROXY:',final)
SUB=submit(CHAMP['prediction_file'],KNOWN,TEMPLATE,OUT/'submission_g12_deep.tif')
print('Validated submission:',SUB)
REPORT={"feature_version":MAN['stamp']['version'],"feature_count":len(MAN['names']),
 "ablation":ROWS,"champion_by_validation_proxy":CHAMP['model'],"geographic_test_proxy":final,
 "evidence_atlas":ATLAS,"submission":SUB,
 "warning":"Not official DTI: private ground-truth new faults are unavailable."}
(OUT/'report.json').write_text(json.dumps(REPORT,indent=2))
print('Report:',OUT/'report.json')

## 06 — Interpreting the result

- **Score improves + candidate maps plausible:** prioritize the winning physical-feature family and controlled topology-aware postprocessing.
- **PR-AUC improves but DTI worsens:** investigate diffuse probability mass, over-wide traces and spatial calibration; do not add models blindly.
- **Strong false structures along ridges or DEM seams:** inspect `*_coverage`, directional asymmetry and the feature atlas; stratify by DEM presence.
- **Spatial test drops sharply:** geological regime shift is more important than added features. Try region-aware experts next.

**DrivenData:** https://www.drivendata.org/competitions/306/ — upload `submission_g12_deep.tif` only after validating the local plot and metadata. Respect daily score limits. Save the score together with `report.json`, Git commit and configuration.